# Behavior bout GIF viewer

A visual check that the behavior dictionary lines up with the video. For each bout:

1. take its start/stop (play-indexed ms) **from the behavior dictionary**
2. go **backwards through the timestamps array** (`play_indexed_array`) to find the frames
3. add padding, given **in seconds** and converted to frames through the same timestamps
4. read those frames from the mp4 and write a GIF, with the behavior label drawn **only on the frames inside the bout**

**How to read a GIF:** the yellow label should appear as the subject's nose reaches the other mouse and disappear as it pulls away. The header shows the frame number and its time on the ephys clock.

Frame rate is never used to *find* frames, because the camera stamps frames unevenly, so `frame = time * fps` drifts. `FPS` below only sets how fast the GIF plays.

Code: `vid_helper_fxns/behavior_gif.py`. Tests: `pytest vid_helper_fxns/tests`. **Run with the working directory at the repo root.**

In [ ]:
import os, pickle
from IPython.display import Image, display

# notebooks must run from the repo root (see README); step up if launched from this folder
while not os.path.isdir("spike") and os.getcwd() != "/":
    os.chdir("..")
print("working dir:", os.getcwd())

from vid_helper_fxns import behavior_gif as bg

## Inputs

In [ ]:
VIDEO_DIR      = os.path.expanduser("~/Library/CloudStorage/Dropbox/Cum_SocialMemEphys_pilot2/SLEAP/same_diff_mp4s")
TIMESTAMPS     = "pilot2/same_diff_phase2/timestamps_dict_diff_p2.json"
BEHAVIOR_DICT  = "pilot2/same_diff_phase2/behavior_dict_diff_p2_iti05_bout05.pkl"
FPS            = 15     # GIF playback speed only; 15 = real time
PAD_S          = 2.0    # seconds of video before and after each bout
OUT_DIR        = "vid_helper_fxns/bout_gifs"   # gitignored

with open(BEHAVIOR_DICT, "rb") as f:
    behavior = pickle.load(f)
timestamps = bg.load_timestamps(TIMESTAMPS)
print(len(behavior), "recordings:", sorted(behavior))

## Check the videos match their timestamps

The timestamps were recorded for the raw `.h264`; the mp4 is a converted copy. If a conversion dropped or added frames, every GIF from that video would be offset, so check first.

In [ ]:
for rec in sorted(behavior):
    path = bg.video_for(rec, VIDEO_DIR)
    if not os.path.exists(path):
        print(f"{rec:24s} video not found"); continue
    try:
        n_vid, n_ts = bg.check_frame_count(path, timestamps[bg.recording_base(rec)])
        print(f"{rec:24s} OK   {n_vid} frames")
    except ValueError as e:
        print(f"{rec:24s} MISMATCH  {e}")

## One random bout

In [ ]:
def show(rec, event, idx, **kw):
    info = bg.make_bout_gif(behavior, timestamps, bg.video_for(rec, VIDEO_DIR), rec, event, idx,
                            pad_s=PAD_S, fps=FPS, out_dir=OUT_DIR, **kw)
    f = info["frames"]
    print(f"{rec}  {event} #{idx}   bout {info['bout_s'][0]:.3f}-{info['bout_s'][1]:.3f} s"
          f"   frames {f['event_start']}-{f['event_stop']} (+{PAD_S:g} s pad: {f['first']}-{f['last']})"
          + ("   TRUNCATED" if info["truncated"] else ""))
    display(Image(filename=info["gif"]))
    return info

rec, event, idx = bg.random_bouts(behavior, n=1, seed=0)[0]
_ = show(rec, event, idx)

## A random 10

Change `seed` for a different sample; the same seed always gives the same 10.

In [ ]:
for rec, event, idx in bg.random_bouts(behavior, n=10, seed=1):
    show(rec, event, idx)

## A specific bout

Filter by recording and/or event, or pick an exact index.

In [ ]:
REC, EVENT = "12_diff_p2_merged.rec", "novel_c"
print(len(behavior[REC][EVENT]), "bouts of", EVENT, "in", REC)
_ = show(REC, EVENT, 0)